# L-DeepONet - Shinnecock Inlet

In [ ]:
import os
import sys
import tensorflow as tf
tf.keras.backend.set_floatx('float64') 


import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator, ScalarFormatter, FormatStrFormatter
from matplotlib import ticker as mtick
import cmocean

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error as mae
import joblib

from IPython.display import display
import matplotlib.ticker as ticker
from matplotlib import rcParams
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import netCDF4 as nc
from tqdm import tqdm

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.patches as mpatches

plt.rc('font', family='serif')
plt.rcParams.update({'font.size': 20,
                     'lines.linewidth': 2,
                     'lines.markersize':10,
                     'axes.labelsize': 16, # fontsize for x and y labels (was 10)
                     'axes.titlesize': 20,
                     'xtick.labelsize': 16,
                     'ytick.labelsize': 16,
                     'legend.fontsize': 16,
                     'axes.linewidth': 2})

import itertools
colors = itertools.cycle(['r','g','b','m','y','c'])
markers = itertools.cycle(['p','d','o','^','s','x',]) #'D','H','v','*'])

from pathlib import Path
try:
    work_dir.exists()
except NameError:
    curr_dir = Path().resolve()
    work_dir = curr_dir.parent  

scripts_dir = work_dir / "src"
sys.path.append(str(scripts_dir.absolute()))

# ae_model_dir = work_dir / "saved_models"
# ldon_model_dir = work_dir / "saved_models"
# scalers_dir = work_dir / "scalers"
ae_model_dir = work_dir / "saved_models_ldon"
ldon_model_dir = work_dir / "saved_models_ldon"
scalers_dir = work_dir / "scalers_ldon"

#Download and copy your data to this folder
ROOT_DIR = work_dir.parents[1]
SHINNECOCK_DATA_DIR = (
    ROOT_DIR
    / 'data'
    / 'PRJ-5716'
    / 'Simulation--2d-adcirc-simulation-of-tidal-flow-in-shinnecock-inlet-ny-parameterized-by-bottom-friction-coefficient'
    / 'data'
    / 'Model--adcirc-model'
    / 'data'
)
data_dir = SHINNECOCK_DATA_DIR


import ldon 
import autoencoder as ae
import data_loader as dl 
import processing_utils as pu

from importlib import reload as reload

%matplotlib inline

## Specify training configuraiton for scaling purposes and testing cases for visualization

In [ ]:
# Training data for scaling
train_days = [15., 30.]
param_list_train = [0.003, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05]
scaling = True

# Test data
test_days = [5., 50.]
param_list_test = [0.0025, 0.00275, 0.003, 0.005, 0.0075, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05, 0.075, 0.1]
# unseen_param_list_test = [0.0025, 0.015, 0.1]
unseen_param_loc = [0,6,-1]
window_size=5

## Load Models

In [ ]:
# Load H Models
ae_model_path_h = str(ae_model_dir) + '/AE_h'
ae_model_h,_ = ae.load_model(ae_model_path_h, comp = False)
ldon_model_path_h = str(ldon_model_dir) + '/LDON_h'
ldon_model_h = tf.keras.models.load_model(ldon_model_path_h)

# Load U Models
ae_model_path_u = str(ae_model_dir) + '/AE_u'
ae_model_u,_ = ae.load_model(ae_model_path_u, comp = False)
ldon_model_path_u = str(ldon_model_dir) + '/LDON_u'
ldon_model_u = tf.keras.models.load_model(ldon_model_path_u)

# Load V Models
ae_model_path_v = str(ae_model_dir) + '/AE_v'
ae_model_v,_ = ae.load_model(ae_model_path_v, comp = False)
ldon_model_path_v = str(ldon_model_dir) + '/LDON_v'
ldon_model_v = tf.keras.models.load_model(ldon_model_path_v)

### Print Model Summary

In [ ]:
# Print AE model summary
ae_model_u.summary()
ae_model_u.summary()
ae_model_v.summary()

In [ ]:
# Print L-DeepONet model summary
ldon_model_u.summary()
ldon_model_u.summary()
ldon_model_v.summary()

## Load Scalers

In [ ]:
scaler_h = joblib.load(str(scalers_dir)+'/ae_scaler_h.save')
scaler_u = joblib.load(str(scalers_dir)+'/ae_scaler_u.save')
scaler_v = joblib.load(str(scalers_dir)+'/ae_scaler_v.save')

t_scaler_h = joblib.load(str(scalers_dir)+'/t_scaler_h.save') 
b_scaler_h = joblib.load(str(scalers_dir)+'/b_scaler_h.save') 
ls_scaler_h = joblib.load(str(scalers_dir)+'/ls_scaler_h.save') 

t_scaler_u = joblib.load(str(scalers_dir)+'/t_scaler_u.save') 
b_scaler_u = joblib.load(str(scalers_dir)+'/b_scaler_u.save') 
ls_scaler_u = joblib.load(str(scalers_dir)+'/ls_scaler_u.save')

t_scaler_v = joblib.load(str(scalers_dir)+'/t_scaler_v.save') 
b_scaler_v = joblib.load(str(scalers_dir)+'/b_scaler_v.save') 
ls_scaler_v = joblib.load(str(scalers_dir)+'/ls_scaler_v.save') 

## Load test data

In [ ]:
Np = len(param_list_test)

mesh = dl.load_mesh(data_dir/"cf0025")
snaps = [np.where(mesh['t']/60/60/24==test_days[0])[0][0], np.where(mesh['t']/60/60/24==test_days[1])[0][0]]
Nt_snaps = mesh['t'][snaps[0]:snaps[1]].shape[0]

data_dict = {}
for inx, param in enumerate(param_list_test):
    dirname = 'cf'+str(param).split('.')[1];
    data_dict[inx] = dl.load_variables(data_dir/dirname)

Nn = mesh['nodes'].shape[0]
Ne = mesh['triangles'].shape[0]
Nt = mesh['t'].shape[0]


In [ ]:
dataset_h = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_u = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_v = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed

key = 'h'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_h = np.vstack((dataset_h, snap))

key = 'u'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_u = np.vstack((dataset_u, snap))

key = 'v'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_v = np.vstack((dataset_v, snap))
    
dataset_u_resh = np.reshape(dataset_u,(Np,Nt,Nn))
dataset_u_crop = dataset_u_resh[:,snaps[0]:snaps[1],:]
dataset_u = np.reshape(dataset_u_crop,(Np*Nt_snaps,Nn))

dataset_h_resh = np.reshape(dataset_h,(Np,Nt,Nn))
dataset_h_crop = dataset_h_resh[:,snaps[0]:snaps[1],:]
dataset_h = np.reshape(dataset_h_crop,(Np*Nt_snaps,Nn))

dataset_v_resh = np.reshape(dataset_v,(Np,Nt,Nn))
dataset_v_crop = dataset_v_resh[:,snaps[0]:snaps[1],:]
dataset_v = np.reshape(dataset_v_crop,(Np*Nt_snaps,Nn))

if scaling:
    data_scaled_h = scaler_h.transform(dataset_h)
    data_scaled_u = scaler_u.transform(dataset_u)
    data_scaled_v = scaler_v.transform(dataset_v)

dataset_r_h = np.reshape(dataset_h,(Np,Nt_snaps,Nn))
dataset_r_u = np.reshape(dataset_u,(Np,Nt_snaps,Nn))
dataset_r_v = np.reshape(dataset_v,(Np,Nt_snaps,Nn))


In [ ]:
data_ls_h = ae_model_h.encoder(data_scaled_h)
latent_dim_h = data_ls_h.shape[1]
data_ls_h = np.array(np.reshape(data_ls_h,(Np,Nt_snaps,latent_dim_h)))

data_ls_u = ae_model_u.encoder(data_scaled_u)
latent_dim_u = data_ls_u.shape[1]
data_ls_u = np.array(np.reshape(data_ls_u,(Np,Nt_snaps,latent_dim_u)))

data_ls_v = ae_model_v.encoder(data_scaled_v)
latent_dim_v = data_ls_v.shape[1]
data_ls_v = np.array(np.reshape(data_ls_v,(Np,Nt_snaps,latent_dim_v)))

In [ ]:
snaps = [np.where(mesh['t']/60/60/24==test_days[0])[0][0], np.where(mesh['t']/60/60/24==test_days[1]-window_size)[0][0]]
Nt_snaps = mesh['t'][snaps[0]:snaps[1]].shape[0]

dataset_h = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_u = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_v = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed

key = 'h'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_h = np.vstack((dataset_h, snap))

key = 'u'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_u = np.vstack((dataset_u, snap))

key = 'v'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_v = np.vstack((dataset_v, snap))
    
dataset_u_resh = np.reshape(dataset_u,(Np,Nt,Nn))
dataset_u_crop = dataset_u_resh[:,snaps[0]:snaps[1],:]
dataset_u = np.reshape(dataset_u_crop,(Np*Nt_snaps,Nn))

dataset_h_resh = np.reshape(dataset_h,(Np,Nt,Nn))
dataset_h_crop = dataset_h_resh[:,snaps[0]:snaps[1],:]
dataset_h = np.reshape(dataset_h_crop,(Np*Nt_snaps,Nn))

dataset_v_resh = np.reshape(dataset_v,(Np,Nt,Nn))
dataset_v_crop = dataset_v_resh[:,snaps[0]:snaps[1],:]
dataset_v = np.reshape(dataset_v_crop,(Np*Nt_snaps,Nn))

if scaling:
    data_scaled_h = scaler_h.transform(dataset_h)
    data_scaled_u = scaler_u.transform(dataset_u)
    data_scaled_v = scaler_v.transform(dataset_v)

dataset_r_h = np.reshape(dataset_h,(Np,Nt_snaps,Nn))
dataset_r_u = np.reshape(dataset_u,(Np,Nt_snaps,Nn))
dataset_r_v = np.reshape(dataset_v,(Np,Nt_snaps,Nn))

In [ ]:
window_size=5
b_data_h, t_data_h, target_data_h = pu.latent_stacked_param_hard_windows(param_list_test,data_ls_h,snaps[0],snaps[1],data_dir)
b_data_u, t_data_u, target_data_u = pu.latent_stacked_param_hard_windows(param_list_test,data_ls_u,snaps[0],snaps[1],data_dir)
b_data_v, t_data_v, target_data_v = pu.latent_stacked_param_hard_windows(param_list_test,data_ls_v,snaps[0],snaps[1],data_dir)


In [ ]:
if scaling is True:
    t_data_h = t_data_h/t_scaler_h
    b_data_h[:,1:] = b_scaler_h.transform(b_data_h[:,1:])
    target_data_h = ls_scaler_h.transform(target_data_h)

    t_data_u = t_data_u/t_scaler_u
    b_data_u[:,1:] = b_scaler_u.transform(b_data_u[:,1:])
    target_data_u = ls_scaler_u.transform(target_data_u)

    t_data_v = t_data_v/t_scaler_v
    b_data_v[:,1:] = b_scaler_v.transform(b_data_v[:,1:])
    target_data_v = ls_scaler_v.transform(target_data_v)

In [ ]:
b_data_h = np.reshape(b_data_h,(Np,int((Nt_snaps)/window_size),1+latent_dim_h+window_size*75))
target_data_h = np.reshape(target_data_h,(Np,Nt_snaps,latent_dim_h))

b_data_u = np.reshape(b_data_u,(Np,int((Nt_snaps)/window_size),1+latent_dim_u+window_size*75))
target_data_u = np.reshape(target_data_u,(Np,Nt_snaps,latent_dim_u))

b_data_v = np.reshape(b_data_v,(Np,int((Nt_snaps)/window_size),1+latent_dim_v+window_size*75))
target_data_v = np.reshape(target_data_v,(Np,Nt_snaps,latent_dim_v))


### Predictions 1

In [ ]:
res_h = np.zeros((Np, Nt_snaps, latent_dim_h))

for p in tqdm(range(Np)):
    b_ic = b_data_h[p, 0, 1:latent_dim_h+1].copy() 
    res_h[p,0,:] = b_ic
    count = 0
    for i in range(0, int((Nt_snaps)/window_size)):
        if (count+window_size) < (Nt_snaps):
            b_input = b_data_h[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:latent_dim_h+1] = b_ic
            t_input = t_data_h
            
            prediction = ldon_model_h((b_input, t_input)).numpy().reshape(window_size,latent_dim_h)
            res_h[p,count+1:count+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
            count = count + window_size
        else:
            diff = (Nt_snaps) - count - 1
            b_input = b_data_h[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:latent_dim_h+1] = b_ic
            t_input = t_data_h
            
            prediction = ldon_model_h((b_input, t_input)).numpy().reshape(window_size,latent_dim_h)
            res_h[p,count+1:count+diff+1] = prediction[:diff,:]

res_h = ls_scaler_h.inverse_transform(res_h.reshape(int(Np*(Nt_snaps)/window_size),int(latent_dim_h*window_size)))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_h = ae_model_h.decoder(res_h.reshape(int(Np*(Nt_snaps)),int(latent_dim_h))).numpy()
full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))

# Reshape data for analysis and visualization
full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps, Nn))


In [ ]:
res_u = np.zeros((Np, Nt_snaps, latent_dim_u))

for p in tqdm(range(Np)):
    b_ic = b_data_u[p, 0, 1:latent_dim_u+1].copy() 
    res_u[p,0,:] = b_ic
    count = 0
    for i in range(0, int((Nt_snaps)/window_size)):
        if (count+window_size) < (Nt_snaps):
            b_input = b_data_u[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:latent_dim_u+1] = b_ic
            t_input = t_data_u
            
            prediction = ldon_model_u((b_input, t_input)).numpy().reshape(window_size,latent_dim_u)
            res_u[p,count+1:count+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
            count = count + window_size
        else:
            diff = (Nt_snaps) - count - 1
            b_input = b_data_u[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:latent_dim_u+1] = b_ic
            t_input = t_data_u
            
            prediction = ldon_model_u((b_input, t_input)).numpy().reshape(window_size,latent_dim_u)
            res_u[p,count+1:count+diff+1] = prediction[:diff,:]

res_u = ls_scaler_u.inverse_transform(res_u.reshape(int(Np*(Nt_snaps)/window_size),int(latent_dim_u*window_size)))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_u = ae_model_u.decoder(res_u.reshape(int(Np*(Nt_snaps)),int(latent_dim_u))).numpy()
full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))

# Reshape data for analysis and visualization
full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps, Nn))




In [ ]:
res_v = np.zeros((Np, Nt_snaps, latent_dim_v))

for p in tqdm(range(Np)):
    b_ic = b_data_v[p, 0, 1:latent_dim_v+1].copy() 
    res_v[p,0,:] = b_ic
    count = 0
    for i in range(0, int((Nt_snaps)/window_size)):
        if (count+window_size) < (Nt_snaps):
            b_input = b_data_v[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:latent_dim_v+1] = b_ic
            t_input = t_data_v
            
            prediction = ldon_model_v((b_input, t_input)).numpy().reshape(window_size,latent_dim_v)
            res_v[p,count+1:count+window_size+1] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
            count = count + window_size
        else:
            diff = (Nt_snaps) - count - 1
            b_input = b_data_v[p, i, :].reshape(1,-1).copy()
            b_input[0, 1:latent_dim_v+1] = b_ic
            t_input = t_data_v
            
            prediction = ldon_model_v((b_input, t_input)).numpy().reshape(window_size,latent_dim_v)
            res_v[p,count+1:count+diff+1] = prediction[:diff,:]

res_v = ls_scaler_v.inverse_transform(res_v.reshape(int(Np*(Nt_snaps)/window_size),int(latent_dim_v*window_size)))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_v = ae_model_v.decoder(res_v.reshape(int(Np*(Nt_snaps)),int(latent_dim_v))).numpy()
full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))

# Reshape data for analysis and visualization
full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps, Nn))


In [ ]:
full_res_r_h.shape

In [ ]:
np.savez('ldon_output.npz', h_data=full_res_r_h, u_data=full_res_r_u, v_data=full_res_r_v)

In [ ]:
mask_num = 0
rmse_h = np.zeros((Np,Nt_snaps))
rmse_u = np.zeros((Np,Nt_snaps))
rmse_v = np.zeros((Np,Nt_snaps))

max_array_h = np.zeros(Np)
max_array_u = np.zeros(Np)
max_array_v = np.zeros(Np)

n_rmse_h = np.zeros((Np,Nt_snaps))
n_rmse_u = np.zeros((Np,Nt_snaps))
n_rmse_v = np.zeros((Np,Nt_snaps))

for p, param in enumerate(param_list_test[0:]):
    rmse_h[p] = np.sqrt(np.mean((full_res_r_h[p, :, mask_num:] - dataset_r_h[p, :, mask_num:]) ** 2, axis=1))*100
    rmse_u[p] = np.sqrt(np.mean((full_res_r_u[p, :, mask_num:] - dataset_r_u[p, :, mask_num:]) ** 2, axis=1))*100
    rmse_v[p] = np.sqrt(np.mean((full_res_r_v[p, :, mask_num:] - dataset_r_v[p, :, mask_num:]) ** 2, axis=1))*100

    max_array_h[p] = (np.abs(np.max(dataset_r_h[p, :, mask_num:])-np.min(dataset_r_h[p,  :, mask_num:])))*100
    max_array_u[p] = (np.abs(np.max(dataset_r_u[p, :, mask_num:])-np.min(dataset_r_u[p,  :, mask_num:])))*100
    max_array_v[p] = (np.abs(np.max(dataset_r_v[p, :, mask_num:])-np.min(dataset_r_v[p,  :, mask_num:])))*100

    n_rmse_h[p] = rmse_h[p]/max_array_h[p]
    n_rmse_u[p] = rmse_u[p]/max_array_u[p]
    n_rmse_v[p] = rmse_v[p]/max_array_v[p]
    

#### ACC and RMSE Table

In [ ]:
##### ACC
P = dataset_r_h.shape[0]
MM = dataset_r_h.shape[1]
N = dataset_r_h.shape[2]

ACC_h = np.zeros(dataset_r_h.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_h[p,j,:]*100)/N
        Tj = np.sum(dataset_r_h[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_h[p,j,:]*100-Tj)*(full_res_r_h[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_h[p,j,:]*100-Tj)**2)*np.sum((full_res_r_h[p,j,:]*100-Pj)**2))
    ACC_h[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_h.shape[2]):
            
print('ACC:'+str(list(ACC_h)))
print('RMSE:'+str(list(np.mean(rmse_h,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_h,axis=1))))


In [ ]:
##### ACC

P = dataset_r_u.shape[0]
MM = dataset_r_u.shape[1]
N = dataset_r_u.shape[2]

ACC_u = np.zeros(dataset_r_u.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_u[p,j,:]*100)/N
        Tj = np.sum(dataset_r_u[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_u[p,j,:]*100-Tj)*(full_res_r_u[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_u[p,j,:]*100-Tj)**2)*np.sum((full_res_r_u[p,j,:]*100-Pj)**2))
    ACC_u[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_u.shape[2]):
            
print('ACC:'+str(list(ACC_u)))
print('RMSE:'+str(list(np.mean(rmse_u,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_u,axis=1))))


In [ ]:
##### ACC

P = dataset_r_v.shape[0]
MM = dataset_r_v.shape[1]
N = dataset_r_v.shape[2]

ACC_v = np.zeros(dataset_r_v.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_v[p,j,:]*100)/N
        Tj = np.sum(dataset_r_v[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_v[p,j,:]*100-Tj)*(full_res_r_v[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_v[p,j,:]*100-Tj)**2)*np.sum((full_res_r_v[p,j,:]*100-Pj)**2))
    ACC_v[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_v.shape[2]):
            
print('ACC:'+str(list(ACC_v)))
print('RMSE:'+str(list(np.mean(rmse_v,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_v,axis=1))))


### Predictions 2

In [ ]:
# window_size = 5
# day_window = 5.
# day_total = test_days[1]-test_days[0]

# n_windows = int(day_total/day_window)

# snap_windows = int(Nt_snaps/day_total*day_window)

# rmse_array_h = np.zeros((n_windows, len(param_list_test)))
# rmse_array_u = np.zeros((n_windows, len(param_list_test)))
# rmse_array_v = np.zeros((n_windows, len(param_list_test)))

# max_array_h = np.zeros((n_windows, len(param_list_test)))
# max_array_u = np.zeros((n_windows, len(param_list_test)))
# max_array_v = np.zeros((n_windows, len(param_list_test)))

In [ ]:
# current_window = 0

# for w in tqdm(range(n_windows)):
#     res_h = np.zeros((Np, snap_windows, latent_dim_h))
#     res_u = np.zeros((Np, snap_windows, latent_dim_u))
#     res_v = np.zeros((Np, snap_windows, latent_dim_v))
    
#     for p in range(Np):

#         b_ic_h = ls_scaler_h.transform(np.expand_dims(data_ls_h[p, current_window, :],0))[0]#b_ic_data_h[p, current_window,:].copy()#
#         res_h[p,0,:] = b_ic_h
        
#         b_ic_u = ls_scaler_u.transform(np.expand_dims(data_ls_u[p, current_window, :],0))[0]
#         res_u[p,0,:] = b_ic_u
        
#         b_ic_v = ls_scaler_v.transform(np.expand_dims(data_ls_v[p, current_window, :],0))[0]
#         res_v[p,0,:] = b_ic_v
        
#         for i in range(0, snap_windows, window_size):
#             temp_i = i + current_window

#             if i+window_size+1 < snap_windows:
                
#                 par_input_h = b_par_data_h[p, temp_i+1:temp_i+window_size+1, :]   
#                 ic_input_h = np.tile(np.expand_dims(b_ic_h,1),window_size).T 
#                 bc_input_h = b_bc_data_h[p, temp_i+1:temp_i+window_size+1, :]  
#                 t_input_h = np.expand_dims(t_data_h[0:window_size],1)  
#                 prediction_h = ldon_model_h([par_input_h, ic_input_h, bc_input_h, t_input_h]).numpy()
#                 res_h[p,i+1:i+window_size+1] = prediction_h 
#                 b_ic_h = prediction_h[-1].copy()   # Update initial condition for the next iteration

#                 par_input_u = b_par_data_u[p, temp_i+1:temp_i+window_size+1, :]    
#                 ic_input_u = np.tile(np.expand_dims(b_ic_u,1),window_size).T 
#                 bc_input_u = b_bc_data_u[p, temp_i+1:temp_i+window_size+1, :]   
#                 t_input_u = np.expand_dims(t_data_u[0:window_size],1)  
#                 prediction_u = ldon_model_u([par_input_u, ic_input_u, bc_input_u, t_input_u]).numpy()
#                 res_u[p,i+1:i+window_size+1] = prediction_u 
#                 b_ic_u = prediction_u[-1]   # Update initial condition for the next iteration

#                 par_input_v = b_par_data_v[p, temp_i+1:temp_i+window_size+1, :]    
#                 ic_input_v = np.tile(np.expand_dims(b_ic_v,1),window_size).T 
#                 bc_input_v = b_bc_data_v[p, temp_i+1:temp_i+window_size+1, :]   
#                 t_input_v = np.expand_dims(t_data_v[0:window_size],1)  
#                 prediction_v = ldon_model_v([par_input_v, ic_input_v, bc_input_v, t_input_v]).numpy()
#                 res_v[p,i+1:i+window_size+1] = prediction_v 
#                 b_ic_v = prediction_v[-1]   # Update initial condition for the next iteration
#             else:
#                 diff = snap_windows - i - 1
                
#                 par_input_h = b_par_data_h[p, temp_i+1:temp_i+1+diff, :]    
#                 ic_input_h = np.tile(np.expand_dims(b_ic_h,1),diff).T  
#                 bc_input_h = b_bc_data_h[p, temp_i+1:temp_i+1+diff, :]   
#                 t_input_h = np.expand_dims(t_data_h[0:diff],1)  
#                 prediction_h = ldon_model_h([par_input_h, ic_input_h, bc_input_h, t_input_h]).numpy()
#                 res_h[p,i+1:i+diff+1] = prediction_h 
                
#                 par_input_u = b_par_data_u[p, temp_i+1:temp_i+1+diff, :]    
#                 ic_input_u = np.tile(np.expand_dims(b_ic_u,1),diff).T 
#                 bc_input_u = b_bc_data_u[p, temp_i+1:temp_i+1+diff, :]   
#                 t_input_u = np.expand_dims(t_data_u[0:diff],1)  
#                 prediction_u = ldon_model_u([par_input_u, ic_input_u, bc_input_u, t_input_u]).numpy()
#                 res_u[p,i+1:i+diff+1] = prediction_u 
                
#                 par_input_v = b_par_data_v[p, temp_i+1:temp_i+1+diff, :]    
#                 ic_input_v = np.tile(np.expand_dims(b_ic_v,1),diff).T 
#                 bc_input_v = b_bc_data_v[p, temp_i+1:temp_i+1+diff, :]   
#                 t_input_v = np.expand_dims(t_data_v[0:diff],1)  
#                 prediction_v = ldon_model_v([par_input_v, ic_input_v, bc_input_v, t_input_v]).numpy()
#                 res_v[p,i+1:i+diff+1] = prediction_v 

#     res_h = ls_scaler_h.inverse_transform(res_h.reshape(Np*snap_windows,latent_dim_h))
#     # res_h = ls_scaler_h.inverse_transform(res_h)        
#     full_res_h = ae_model_h.decoder(res_h).numpy()
#     full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))
#     full_res_r_h = np.reshape(full_res_h, (Np, snap_windows, Nn))

#     res_u = ls_scaler_u.inverse_transform(res_u.reshape(Np*snap_windows,latent_dim_u))
#     # res_u = ls_scaler_u.inverse_transform(res_u)        
#     full_res_u = ae_model_u.decoder(res_u).numpy()
#     full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))
#     full_res_r_u = np.reshape(full_res_u, (Np, snap_windows, Nn))

#     res_v = ls_scaler_v.inverse_transform(res_v.reshape(Np*snap_windows,latent_dim_v))
#     # res_v = ls_scaler_v.inverse_transform(res_v)        
#     full_res_v = ae_model_v.decoder(res_v).numpy()
#     full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))
#     full_res_r_v = np.reshape(full_res_v, (Np, snap_windows, Nn))

    
#     for p in range(Np):

#         rmse_array_h[w, p] = np.mean(np.sqrt(np.mean((full_res_r_h[p, :] - dataset_r_h[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))
#         max_array_h[w, p] = (np.abs(np.max(dataset_r_h[p, current_window:snap_windows+current_window])-np.min(dataset_r_h[p, current_window:snap_windows+current_window])))
        
#         rmse_array_u[w, p] = np.mean(np.sqrt(np.mean((full_res_r_u[p, :] - dataset_r_u[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))
#         max_array_u[w, p] = (np.abs(np.max(dataset_r_u[p, current_window:snap_windows+current_window])-np.min(dataset_r_u[p, current_window:snap_windows+current_window])))
        
#         rmse_array_v[w, p] = np.mean(np.sqrt(np.mean((full_res_r_v[p, :] - dataset_r_v[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))
#         max_array_v[w, p] = (np.abs(np.max(dataset_r_v[p, current_window:snap_windows+current_window])-np.min(dataset_r_v[p, current_window:snap_windows+current_window])))

#     current_window += snap_windows



### Predictions 3

In [ ]:
# window_size = 1

In [ ]:
# if scaling:
#     zero_h = scaler_h.transform(np.zeros((1,dataset_h.shape[1])))
#     zero_u = scaler_u.transform(np.zeros((1,dataset_u.shape[1])))
#     zero_v = scaler_v.transform(np.zeros((1,dataset_v.shape[1])))

#     zero_ic_h = ls_scaler_h.transform(ae_model_h.encoder(zero_h))
#     zero_ic_u = ls_scaler_u.transform(ae_model_u.encoder(zero_u))
#     zero_ic_v = ls_scaler_v.transform(ae_model_v.encoder(zero_v))

In [ ]:
# snaps_crop = [np.where(mesh['t']/60/60/24==45.)[0][0], np.where(mesh['t']/60/60/24==55.)[0][0]]
# Nt_snaps_crop = mesh['t'][snaps_crop[0]:snaps_crop[1]].shape[0]
# snaps_crop -= snaps[0]


In [ ]:
# res_h = np.zeros((Np, Nt_snaps_crop, latent_dim_h))

# for p in tqdm(range(Np)):
#     b_ic = zero_ic_h[0] 
#     res_h[p,0,:] = b_ic
#     for i in range(snaps_crop[0],snaps_crop[1], window_size):
#         if i+window_size < Nt_snaps:
#             par_input = b_par_data_h[p, i+1:i+window_size+1, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
#             bc_input = b_bc_data_h[p, i+1:i+window_size+1, :] 
#             t_input = np.expand_dims(t_data_h[0:window_size],1)  
    
#             prediction = ldon_model_h([par_input, ic_input, bc_input, t_input])
#             res_h[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#         else:
#             diff = Nt_snaps - i - 1
#             par_input = b_par_data_h[p, i+1:i+1+diff, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
#             bc_input = b_bc_data_h[p, i+1:i+1+diff, :] 
#             t_input = np.expand_dims(t_data_h[0:diff],1)  

#             prediction = ldon_model_h([par_input, ic_input, bc_input, t_input])
#             res_h[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction
            

# res_h = ls_scaler_h.inverse_transform(res_h.reshape(Np*Nt_snaps_crop,latent_dim_h))

# #### Post-processing ####

# # Decode the latent representation to get the full resolution
# full_res_h = ae_model_h.decoder(res_h).numpy()
# full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))

# # Reshape data for analysis and visualization
# full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps_crop, Nn))

In [ ]:
# res_u = np.zeros((Np, Nt_snaps_crop, latent_dim_u))

# for p in tqdm(range(Np)):
#     b_ic = zero_ic_u[0] 
#     res_u[p,0,:] = b_ic
#     for i in range(snaps_crop[0],snaps_crop[1], window_size):
#         if i+window_size < Nt_snaps:
#             par_input = b_par_data_u[p, i+1:i+window_size+1, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
#             bc_input = b_bc_data_u[p, i+1:i+window_size+1, :] 
#             t_input = np.expand_dims(t_data_u[0:window_size],1)  
    
#             prediction = ldon_model_u([par_input, ic_input, bc_input, t_input])
#             res_u[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#         else:
#             diff = Nt_snaps - i - 1
#             par_input = b_par_data_u[p, i+1:i+1+diff, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
#             bc_input = b_bc_data_u[p, i+1:i+1+diff, :] 
#             t_input = np.expand_dims(t_data_u[0:diff],1)  

#             prediction = ldon_model_u([par_input, ic_input, bc_input, t_input])
#             res_u[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

# res_u = ls_scaler_u.inverse_transform(res_u.reshape(Np*Nt_snaps_crop,latent_dim_u))

# #### Post-processing ####

# # Decode the latent representation to get the full resolution
# full_res_u = ae_model_u.decoder(res_u).numpy()
# full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))

# # Reshape data for analysis and visualization
# full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps_crop, Nn))

In [ ]:
# res_v = np.zeros((Np, Nt_snaps_crop, latent_dim_v))

# for p in tqdm(range(Np)):
#     b_ic = zero_ic_v[0] 
#     res_v[p,0,:] = b_ic
#     for i in range(snaps_crop[0],snaps_crop[1], window_size):
#         if i+window_size < Nt_snaps:
#             par_input = b_par_data_v[p, i+1:i+window_size+1, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
#             bc_input = b_bc_data_v[p, i+1:i+window_size+1, :] 
#             t_input = np.expand_dims(t_data_v[0:window_size],1)  
    
#             prediction = ldon_model_v([par_input, ic_input, bc_input, t_input])
#             res_v[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#         else:
#             diff = Nt_snaps - i - 1
#             par_input = b_par_data_v[p, i+1:i+1+diff, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
#             bc_input = b_bc_data_v[p, i+1:i+1+diff, :] 
#             t_input = np.expand_dims(t_data_v[0:diff],1)  

#             prediction = ldon_model_v([par_input, ic_input, bc_input, t_input])
#             res_v[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

# res_v = ls_scaler_v.inverse_transform(res_v.reshape(Np*Nt_snaps_crop,latent_dim_v))

# #### Post-processing ####

# # Decode the latent representation to get the full resolution
# full_res_v = ae_model_v.decoder(res_v).numpy()
# full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))

# # Reshape data for analysis and visualization
# full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps_crop, Nn))

### Predictions 4

In [ ]:
# res_h = np.zeros((Np, Nt_snaps_crop, latent_dim_h))

# for p in tqdm(range(Np)):
#     b_ic = ls_scaler_h.transform(np.expand_dims(data_ls_h[p, snaps_crop[0], :],0))[0]
#     res_h[p,0,:] = b_ic
#     for i in range(snaps_crop[0],snaps_crop[1], window_size):
#         if i+window_size < Nt_snaps:
#             par_input = b_par_data_h[p, i+1:i+window_size+1, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
#             bc_input = b_bc_data_h[p, i+1:i+window_size+1, :] 
#             t_input = np.expand_dims(t_data_h[0:window_size],1)  
    
#             prediction = ldon_model_h([par_input, ic_input, bc_input, t_input])
#             res_h[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#         else:
#             diff = Nt_snaps - i - 1
#             par_input = b_par_data_h[p, i+1:i+1+diff, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
#             bc_input = b_bc_data_h[p, i+1:i+1+diff, :] 
#             t_input = np.expand_dims(t_data_h[0:diff],1)  

#             prediction = ldon_model_h([par_input, ic_input, bc_input, t_input])
#             res_h[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction
            

# res_h = ls_scaler_h.inverse_transform(res_h.reshape(Np*Nt_snaps_crop,latent_dim_h))

# #### Post-processing ####

# # Decode the latent representation to get the full resolution
# full_res_h = ae_model_h.decoder(res_h).numpy()
# full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))

# # Reshape data for analysis and visualization
# full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps_crop, Nn))

In [ ]:
# res_u = np.zeros((Np, Nt_snaps_crop, latent_dim_u))

# for p in tqdm(range(Np)):
#     b_ic = ls_scaler_u.transform(np.expand_dims(data_ls_u[p, snaps_crop[0], :],0))[0]
#     res_u[p,0,:] = b_ic
#     for i in range(snaps_crop[0],snaps_crop[1], window_size):
#         if i+window_size < Nt_snaps:
#             par_input = b_par_data_u[p, i+1:i+window_size+1, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
#             bc_input = b_bc_data_u[p, i+1:i+window_size+1, :] 
#             t_input = np.expand_dims(t_data_u[0:window_size],1)  
    
#             prediction = ldon_model_u([par_input, ic_input, bc_input, t_input])
#             res_u[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#         else:
#             diff = Nt_snaps - i - 1
#             par_input = b_par_data_u[p, i+1:i+1+diff, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
#             bc_input = b_bc_data_u[p, i+1:i+1+diff, :] 
#             t_input = np.expand_dims(t_data_u[0:diff],1)  

#             prediction = ldon_model_u([par_input, ic_input, bc_input, t_input])
#             res_u[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

# res_u = ls_scaler_u.inverse_transform(res_u.reshape(Np*Nt_snaps_crop,latent_dim_u))

# #### Post-processing ####

# # Decode the latent representation to get the full resolution
# full_res_u = ae_model_u.decoder(res_u).numpy()
# full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))

# # Reshape data for analysis and visualization
# full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps_crop, Nn))

In [ ]:
# res_v = np.zeros((Np, Nt_snaps_crop, latent_dim_v))

# for p in tqdm(range(Np)):
#     b_ic = ls_scaler_v.transform(np.expand_dims(data_ls_v[p, snaps_crop[0], :],0))[0]
#     res_v[p,0,:] = b_ic
#     for i in range(snaps_crop[0],snaps_crop[1], window_size):
#         if i+window_size < Nt_snaps:
#             par_input = b_par_data_v[p, i+1:i+window_size+1, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
#             bc_input = b_bc_data_v[p, i+1:i+window_size+1, :] 
#             t_input = np.expand_dims(t_data_v[0:window_size],1)  
    
#             prediction = ldon_model_v([par_input, ic_input, bc_input, t_input])
#             res_v[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
#             b_ic = prediction[-1]  # Update initial condition for the next iteration
#         else:
#             diff = Nt_snaps - i - 1
#             par_input = b_par_data_v[p, i+1:i+1+diff, :]  
#             ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
#             bc_input = b_bc_data_v[p, i+1:i+1+diff, :] 
#             t_input = np.expand_dims(t_data_v[0:diff],1)  

#             prediction = ldon_model_v([par_input, ic_input, bc_input, t_input])
#             res_v[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

# res_v = ls_scaler_v.inverse_transform(res_v.reshape(Np*Nt_snaps_crop,latent_dim_v))

# #### Post-processing ####

# # Decode the latent representation to get the full resolution
# full_res_v = ae_model_v.decoder(res_v).numpy()
# full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))

# # Reshape data for analysis and visualization
# full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps_crop, Nn))